# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

**Sinh viên:** Đặng Đình Đoàn — 2A202602927

Notebook chạy **toàn bộ** bài lab trên Kaggle (GPU T4 x2 hoặc P100) bằng một lần *Save & Run All*:

| Bước | Ô | Nội dung | exp_id |
|---|---|---|---|
| 0 | EDA + kiểm tra | kiểm tra split (S1-S6), phân bố lớp, ảnh mẫu, loss ban đầu ≈ ln 9, overfit 1 batch, ảnh sau augmentation | — |
| 1 | Backbone | 7 backbone cùng công thức nền + DINOv2 linear probe (thưởng) | B01–B08 |
| 2 | Công thức huấn luyện | T00 x 3 seed (đo nhiễu) + 14 ablation một-yếu-tố (trục A–G) + 1 kết hợp | T00–T15 |
| 3 | Suy luận | TTA lật / 5-10 crop / đa tỉ lệ, gộp prob vs logit, dò độ phân giải, ensemble, EMA/soup, temperature scaling, AMP/FP16, gộp BN, độ trễ p50/p95/p99 | I00–I08 |
| 4 | Chung kết | F01 x 3 seed; **test chạy đúng một lần mỗi seed**; mốc T00+I00; `eval.py score` + `eval.py grade` | F01, F02, T00 |
| 5 | Sản phẩm | lệch phân phối, Grad-CAM, ma trận nhầm lẫn, biểu đồ, `results.xlsx`, gói kết quả để nộp | — |

**Mọi lựa chọn (backbone, công thức, phương pháp suy luận) được code tự động quyết định CHỈ dựa trên val**, theo luật
viết sẵn trong `code/experiments.py` (docstring của từng hàm `select_*`), và ghi kèm số liệu căn cứ vào `decisions.json`.
Không bước nào đọc nhãn test trước Bước 4.

**Cách chạy trên Kaggle**
1. *Settings*: Accelerator = **GPU T4 x2** (hoặc P100), **Internet = On** (tải repo, nhãn, trọng số timm).
2. (Khuyến nghị) Thêm dataset chứa `images.zip` của DeepWeeds (Zenodo) qua *Add Input*; nếu không, notebook tự tải từ Zenodo (~490 MB).
3. Chạy thử nhanh trước: đặt `SMOKE = True`, *Run All* (~15–25 phút, kiểm tra toàn bộ pipeline với vài bước train mỗi thí nghiệm, kết quả vào `outputs_smoke/`, KHÔNG dùng để báo cáo).
4. Chạy thật: `SMOKE = False`, **Save Version → Save & Run All (Commit)**. Ước lượng 3–5 giờ trên T4 x2 (tự đo epoch đầu trong log).
5. Tải output: `submission_export.zip` (không có checkpoint) + notebook đã chạy. Giải nén vào `submissions/2A202602927_DangDinhDoan/`.

Nếu phiên bị ngắt: tạo version mới, thêm output của version cũ làm Input và đặt `RESUME_FROM` — các lần chạy đã xong
được bỏ qua, lần chạy dở được chạy tiếp từ checkpoint epoch cuối.

## 0. Cài đặt môi trường

In [ ]:
# ==== Thiết lập (sửa ở đây) ====
SMOKE = False          # True: chạy thử nhanh toàn pipeline (KHÔNG dùng số liệu để báo cáo)
RESUME_FROM = None     # ví dụ "/kaggle/input/<output-version-cu>/outputs" để chạy tiếp phiên bị ngắt
RUN_BONUS_DINOV2 = True
FORCE = {"backbone": None, "recipe": None, "inference": None}   # None = luật tự động trên val

REPO_URL = "https://github.com/Doan0904/K4-Track4-Day2-Deeplearning-Advance.git"
REPO_DIR = "/tmp/lab_repo" if __import__("os").path.isdir("/kaggle") else "lab_repo"
WORK = "/kaggle/working" if __import__("os").path.isdir("/kaggle/working") else "."

In [ ]:
import os, sys, subprocess, platform, time, json, glob, shutil
os.makedirs(os.path.dirname(REPO_DIR) or ".", exist_ok=True)
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)

!pip -q install timm openpyxl

CODE_DIR = glob.glob(f"{REPO_DIR}/submissions/*2A202602927*/code")[0]
sys.path.insert(0, REPO_DIR)      # eval.py gốc (không sửa)
sys.path.insert(0, CODE_DIR)      # code đã hoàn thiện từ starter/

import numpy as np, pandas as pd, torch, torchvision, timm
print("python", platform.python_version(), "| torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| timm", timm.__version__, "| CUDA", torch.version.cuda)
print("GPU:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "KHÔNG CÓ GPU")
print("CPU:", os.cpu_count(), "| code:", CODE_DIR)
!nvidia-smi

In [ ]:
# Test tự viết (CPU, không cần dữ liệu): focal gamma=0 == CE, CutMix lam = diện tích thật, gộp BN, TS, ...
r = subprocess.run([sys.executable, "-m", "unittest", "test_code"], cwd=CODE_DIR, capture_output=True, text=True,
                   env=dict(os.environ, CUDA_VISIBLE_DEVICES=""))
print(r.stderr[-3000:])
assert r.returncode == 0, "test_code thất bại"

### Tải dữ liệu
Ảnh: tìm `images.zip` / thư mục ảnh trong `/kaggle/input`, nếu không có thì tải từ Zenodo; kiểm tra MD5
`b7b30f96d466fba86016aa5a26606e0f`. Nhãn và file chia fold: tải nguyên bản từ GitHub của tác giả (không sửa).
Sau đó giải mã toàn bộ ảnh một lần vào `/tmp/deepweeds_u8.npy` (cache uint8, ~3,4 GB) để DataLoader
không phải giải mã JPEG mỗi epoch (Kaggle chỉ có 4 vCPU). Cache chỉ tăng tốc, ảnh giống hệt ảnh gốc.

In [ ]:
import hashlib, zipfile, urllib.request
DATA = "/tmp/deepweeds" if os.path.isdir("/kaggle") else "data"
IMAGES_DIR, LABELS_DIR = f"{DATA}/images", f"{DATA}/labels"
os.makedirs(IMAGES_DIR, exist_ok=True); os.makedirs(LABELS_DIR, exist_ok=True)
MD5 = "b7b30f96d466fba86016aa5a26606e0f"
FIRST = "20160928-140314-0.jpg"

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

n_have = len(glob.glob(f"{IMAGES_DIR}/*.jpg"))
if n_have < 17509:
    zips = [p for p in glob.glob("/kaggle/input/**/images.zip", recursive=True)]
    dirs = [os.path.dirname(p) for p in glob.glob(f"/kaggle/input/**/{FIRST}", recursive=True)]
    if zips:
        src_zip = zips[0]
    elif dirs and len(glob.glob(f"{dirs[0]}/*.jpg")) >= 17509:
        src_zip = None
        print("Dùng ảnh đã giải nén sẵn trong input:", dirs[0])
        IMAGES_DIR = dirs[0]
    else:
        src_zip = f"{DATA}/images.zip"
        if not os.path.exists(src_zip):
            print("Tải images.zip từ Zenodo ...")
            urllib.request.urlretrieve("https://zenodo.org/records/7939060/files/images.zip?download=1", src_zip)
    if src_zip:
        h = md5(src_zip)
        print("MD5", h)
        assert h == MD5, f"MD5 sai: {h}"
        with zipfile.ZipFile(src_zip) as z:
            for m in z.infolist():
                if m.filename.endswith(".jpg"):
                    m.filename = os.path.basename(m.filename)   # zip giải nén phẳng; gom vào images/
                    z.extract(m, IMAGES_DIR)
print("Số ảnh:", len(glob.glob(f"{IMAGES_DIR}/*.jpg")), "trong", IMAGES_DIR)

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    urllib.request.urlretrieve(f"{BASE}/{name}.csv", f"{LABELS_DIR}/{name}.csv")
!ls -la {LABELS_DIR}; wc -l {LABELS_DIR}/*.csv

In [ ]:
import dataset as D
CACHE = ("/tmp" if os.path.isdir("/kaggle") else DATA) + "/deepweeds_u8.npy"
t0 = time.time()
try:
    D.build_cache(pd.read_csv(f"{LABELS_DIR}/labels.csv")["Filename"], IMAGES_DIR, CACHE, workers=os.cpu_count())
    print(f"cache sẵn sàng ({time.time() - t0:.0f}s):", CACHE)
except OSError as e:  # hết đĩa -> đọc JPEG trực tiếp
    print("Không tạo được cache, đọc JPEG trực tiếp:", e)
    CACHE = None
!df -h /tmp /kaggle/working 2>/dev/null; free -g

In [ ]:
from experiments import Lab
import analysis as AN

OUT = f"{WORK}/outputs_smoke" if SMOKE else f"{WORK}/outputs"
if RESUME_FROM and not os.path.exists(f"{OUT}/decisions.json"):
    shutil.copytree(RESUME_FROM, OUT, dirs_exist_ok=True)
    print("Đã chép kết quả phiên trước từ", RESUME_FROM)
lab = Lab(OUT, IMAGES_DIR, LABELS_DIR, cache_path=CACHE, smoke=SMOKE)
print("OUT =", OUT, "| số GPU dùng song song:", lab.n_gpus, "| SMOKE =", SMOKE)
TIMER = {}
def tic(name): TIMER[name] = time.time(); print(f"=== {name} bắt đầu {time.strftime('%H:%M:%S')}")
def toc(name): print(f"=== {name} xong sau {(time.time() - TIMER[name]) / 60:.1f} phút")

## Bước 0 — EDA và kiểm tra pipeline
Kiểm tra bắt buộc (README mục 2.1): số ảnh mỗi tập/lớp, giao từng cặp tập rỗng, hợp đúng 17.509 ảnh, mọi file tồn tại.
Thêm: nhãn trong subset so với `labels.csv` (chỉ ghi lại chỗ lệch, không sửa CSV theo S1).

In [ ]:
tic("Bước 0")
eda = AN.eda(IMAGES_DIR, LABELS_DIR, OUT)
print("Lệch nhãn subset vs labels.csv:", eda["label_mismatch_vs_labels_csv"])
display(pd.read_csv(f"{OUT}/eda_class_counts.csv", index_col=0))

In [ ]:
from IPython.display import Image as IImage, display
display(IImage(f"{OUT}/figures/eda_class_distribution.png"))
display(IImage(f"{OUT}/figures/eda_samples.png", width=600))

In [ ]:
# Kiểm tra pipeline (slide trang 59, GUIDE mục 1.3): seed cố định; loss ban đầu ≈ ln 9 = 2.197;
# overfit 16 ảnh tới loss ≈ 0; ảnh sau augmentation (giải chuẩn hoá) khớp nhãn; focal gamma=0 == CE.
san = AN.sanity_checks(lab)
assert abs(san["initial_loss"] - np.log(9)) < 0.5, "loss ban đầu lệch xa ln 9"
assert san["overfit_ok"], "không overfit được một batch nhỏ"
display(IImage(f"{OUT}/sanity/overfit_one_batch.png"))
display(IImage(f"{OUT}/sanity/augmentations.png", width=900))
toc("Bước 0")

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE mục 1.4: AdamW, LR backbone 1e-4 / head 1e-3, wd 0,05 không áp cho norm/bias,
warmup 1 epoch + cosine, CE, batch 64, 12 epoch, AMP, RandomResizedCrop 224 + lật ngang; val/test center-crop 224 từ ảnh 256),
cùng seed 0. Chọn checkpoint theo macro-F1 val. Trọng số timm đều chỉ huấn luyện trên ImageNet-1k (tag ghi trong bảng).

In [ ]:
tic("Bước 1")
bt = lab.stage1_backbones(bonus=RUN_BONUS_DINOV2)
display(bt.round(4))
dec_b = lab.select_backbone(force=FORCE["backbone"])
print(json.dumps({k: v for k, v in dec_b.items() if k != "table"}, indent=2, ensure_ascii=False))
toc("Bước 1")

## Bước 2 — Công thức huấn luyện (trục A–G)
`T00` chạy 3 seed để đo nhiễu (std). Mỗi `T01..T14` khác `T00` **đúng một yếu tố**. Sau đó `T15` kết hợp
các yếu tố thắng (luật trong `select_combo`), và chọn công thức chung kết (`select_recipe`).

In [ ]:
tic("Bước 2")
tt = lab.stage2_ablation()
display(tt.round(4))
print("T00:", lab.t00_stats())

In [ ]:
dec_c = lab.select_combo()
print(json.dumps({k: v for k, v in dec_c.items() if k != "candidates"}, indent=2, ensure_ascii=False))
lab.stage2_combo()
dec_r = lab.select_recipe(force=FORCE["recipe"])
print(json.dumps(dec_r, indent=2, ensure_ascii=False))
display(lab.training_table().round(4))
toc("Bước 2")

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Trên **val**, model của công thức đã chọn (seed 0), không huấn luyện lại. Độ trễ: warmup 10 lần, `torch.cuda.synchronize()`
trước/sau, 100 lần đo, p50/p95/p99; batch 1 và 32; FP32 / AMP / FP16; chỉ tính forward (+ tạo view TTA trên GPU),
không tính giải mã/tiền xử lý CPU.

In [ ]:
tic("Bước 3")
inf = lab.stage3_inference()
display(inf.round(4))
dec_i = lab.select_inference(force=FORCE["inference"])
print(json.dumps(dec_i, indent=2, ensure_ascii=False))
print("TTA đổi nhãn (val):", json.load(open(f"{OUT}/inference/tta_flips.json")))
display(pd.read_csv(f"{OUT}/inference/latency.csv").round(3))
toc("Bước 3")

## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Cấu hình đã chốt trên val (`decisions.json`): backbone + công thức + suy luận.
2. Train `F01` với seed 0, 1, 2. Mốc `T00` (+ `I00`) dùng lại 3 seed đã train ở Bước 2.
3. Mỗi model chạy qua test **đúng một lượt** (mọi view cần cho F01 và bản thời gian thực F02 lấy trong cùng lượt đó),
   T khớp trên val của từng seed. Ghi `predictions/F01_seed<k>_test.csv`, `F01_uncal_*`, `F02_*` (nếu F01 dùng TTA), `T00_*`,
   kèm bản `_val.csv`. File test đã có thì KHÔNG chạy lại.
4. Chỉ số tính bằng `eval.py score` và `eval.py grade` (repo gốc, không sửa).

In [ ]:
tic("Bước 4")
res4 = lab.stage4_final()
print(res4)
toc("Bước 4")

In [ ]:
# Gọi lại eval.py gốc bằng dòng lệnh (giống README mục 2.4), để thấy ngay trong notebook:
P, L = f"{OUT}/predictions", LABELS_DIR
!python {REPO_DIR}/eval.py score --pred "{P}/F01_seed*_test.csv" --test-csv {L}/test_subset0.csv --labels {L}/labels.csv --tag F01
!python {REPO_DIR}/eval.py score --pred "{P}/T00_seed*_test.csv" --test-csv {L}/test_subset0.csv --labels {L}/labels.csv --tag T00
p95 = json.load(open(f"{OUT}/inference/latency_final.json"))["realtime_1view_fp32"]["p95"]
!python {REPO_DIR}/eval.py grade --final "{P}/F01_seed*_test.csv" --baseline "{P}/T00_seed*_test.csv" --uncal "{P}/F01_uncal_seed*_test.csv" --final-val "{P}/F01_seed*_val.csv" --val-csv {L}/val_subset0.csv --latency-p95-ms {p95:.3f} --latency-method proper --test-csv {L}/test_subset0.csv --labels {L}/labels.csv

## Bước 5 — Phân tích lỗi, điểm thưởng, sản phẩm
- Ma trận nhầm lẫn test (cộng 3 seed), ảnh bị đoán sai + Grad-CAM (ưu tiên Chinee apple ↔ Snake weed).
- Thưởng: lệch phân phối trên **val** (tối, giảm tương phản, nhiễu, mờ) với ECE trước/sau TS; DINOv2 linear probe (B08).
- `results.xlsx` (Summary, Backbones, Training, Inference, Final, PerClass, Latency + Robustness, Decisions).
- `submission_export/` + `.zip`: mọi file nhỏ để nộp (curves, predictions, figures, logs, eval_results, xlsx), KHÔNG có checkpoint.

In [ ]:
tic("Bước 5")
lab.stage5_bonus()
AN.make_plots(lab)
AN.build_workbook(lab, f"{OUT}/results.xlsx")
for f in ["backbones_tradeoff", "training_ablation", "inference_tradeoff", "confusion_F01_test", "confusion_T00_test",
          "reliability_F01_test", "chinee_snake_confusions"]:
    if os.path.exists(f"{OUT}/figures/{f}.png"):
        display(IImage(f"{OUT}/figures/{f}.png", width=800))
for f in sorted(glob.glob(f"{OUT}/figures/errors_gradcam_*.png")):
    display(IImage(f, width=650))

In [ ]:
display(pd.read_excel(f"{OUT}/results.xlsx", sheet_name="Summary"))
display(pd.read_excel(f"{OUT}/results.xlsx", sheet_name="Final"))
print(open(f"{OUT}/eval_out/eval_stdout.md").read()[-4000:])

In [ ]:
EXPORT = f"{WORK}/submission_export" + ("_smoke" if SMOKE else "")
AN.export(lab, EXPORT)
!ls -la {EXPORT}; du -sh {EXPORT} {OUT}
toc("Bước 5")
print("Tổng thời gian các bước (phút):", {k: round((time.time() - v) / 60, 1) for k, v in TIMER.items()})